# Protein embeddings from a FASTA

Embed proteins with a protein language model (ProtT5, ProstT5, Ankh, ESM2, ESM C or ESM3) and get an
`.h5` file with one vector per protein, or per residue, keyed by protein ID. Use it on its own, or
explore it with ProtSpace: load it in the
[Preparation notebook](https://colab.research.google.com/github/tsenoner/protspace/blob/main/apps/protspace/notebooks/ProtSpace_Preparation.ipynb)
or pass it to `protspace prepare`.

1. **Runtime → Change runtime type → T4 GPU** (on a CPU even small models are slow).
2. In the cell below, pick a model and your FASTA, or leave the FASTA empty to try an example.
3. Run the cell (▶). It downloads the `.h5` when done. Changed a setting? Run it again: the model
   stays loaded, and proteins already embedded are skipped.

📚 [GitHub](https://github.com/tsenoner/protspace) · [protspace.app](https://protspace.app) · [Supported models](https://protspace.app/docs/guide/python-cli#embedder-models)

In [ ]:
# @title Embed proteins { display-mode: "form" }
# @markdown **Model**: [ProtT5](https://huggingface.co/Rostlab/prot_t5_xl_half_uniref50-enc) · [ProstT5](https://huggingface.co/Rostlab/ProstT5_fp16) ·
# @markdown Ankh [base](https://huggingface.co/ElnaggarLab/ankh-base), [large](https://huggingface.co/ElnaggarLab/ankh-large), [3-large](https://huggingface.co/ElnaggarLab/ankh3-large) ·
# @markdown ESM2 [8M](https://huggingface.co/facebook/esm2_t6_8M_UR50D), [35M](https://huggingface.co/facebook/esm2_t12_35M_UR50D), [150M](https://huggingface.co/facebook/esm2_t30_150M_UR50D), [650M](https://huggingface.co/facebook/esm2_t33_650M_UR50D) ·
# @markdown ESM C [300M](https://huggingface.co/biohub/ESMC-300M), [600M](https://huggingface.co/biohub/ESMC-600M) · [ESM3](https://huggingface.co/biohub/esm3-sm-open-v1).
# @markdown Ankh is for non-commercial use; the others are MIT-licensed.
model_name = "Rostlab/prot_t5_xl_half_uniref50-enc"  # @param ["Rostlab/prot_t5_xl_half_uniref50-enc", "Rostlab/ProstT5_fp16", "ElnaggarLab/ankh-base", "ElnaggarLab/ankh-large", "ElnaggarLab/ankh3-large", "facebook/esm2_t6_8M_UR50D", "facebook/esm2_t12_35M_UR50D", "facebook/esm2_t30_150M_UR50D", "facebook/esm2_t33_650M_UR50D", "biohub/ESMC-300M", "biohub/ESMC-600M", "biohub/esm3-sm-open-v1"]
# @markdown One vector per protein (what ProtSpace plots), or one per residue:
embedding_type = "per_prot"  # @param ["per_prot", "per_res"]

# @markdown **Proteins**: a FASTA path in this runtime or in Google Drive (`/content/drive/...`,
# @markdown mounted when needed). Leave it empty to try an example (60 snake-venom proteins),
# @markdown or tick the box to upload your own.
fasta_filename = ""  # @param {type:"string", placeholder:"/content/drive/MyDrive/proteins.fasta"}
upload_fasta = False  # @param {type:"boolean"}
# @markdown Use the UniProt accession in each header as the protein ID (`sp|P12345|NAME` → `P12345`):
should_extract_uniprot_id = True  # @param {type:"boolean"}

# @markdown **Advanced**: the longest sequence to embed (longer ones are skipped and listed), and
# @markdown sequences per batch (halved on GPU out-of-memory).
max_sequence_length = 2000  # @param {type:"integer"}
batch_size = 128  # @param {type:"integer"}

# Run this cell (▶) to embed. Run it again after changing a setting: the model
# stays loaded, and proteins already in the .h5 are skipped.
import logging
import os
import re
import threading
import time
from pathlib import Path

# transformers 4.x (what ESM3's `esm` package installs) imports TensorFlow and
# JAX whenever they are installed, as on Colab, costing seconds and a screen of
# start-up log; this notebook runs on PyTorch only. transformers 5 dropped both,
# and then these do nothing. Set before transformers is first imported.
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("USE_JAX", "0")
os.environ.setdefault("USE_TORCH", "1")
# ProtT5, ProstT5 and Ankh publish only pytorch_model.bin. After loading one,
# transformers 5 starts downloading a converted model.safetensors in the
# background: the same weights again (2.4 GB for ProtT5), never used here.
os.environ["DISABLE_SAFETENSORS_CONVERSION"] = "true"
# transformers 4.x also imports torchao, which Colab ships, and on a CPU runtime
# without Triton it logs "Detected no triton"; nothing here quantizes.
logging.getLogger("torchao.kernel.intmm").setLevel(logging.ERROR)

# Installed via subprocess, not `!pip`: `%%capture` only works on line 1, which
# `# @title` must occupy for Colab to render the form header.
# --- install helper: identical in every ProtSpace Colab notebook (pinned by a test) ---
import importlib
import importlib.metadata as _md
import subprocess
import sys
import tempfile


def _loaded_distributions():
    """{distribution: installed version} for each one an imported module comes from."""
    importlib.invalidate_caches()
    owners = _md.packages_distributions()
    loaded = set(sys.modules)
    found = {}
    for module in [m for m in loaded if "." not in m]:
        for dist in owners.get(module, ()):
            try:
                # A namespace package (`google`) is shared by many distributions:
                # count one only if a subpackage it ships is loaded.
                if getattr(sys.modules[module], "__file__", None) is None:
                    subs = {
                        f.parts[1] for f in _md.distribution(dist).files or ()
                        if len(f.parts) > 2 and f.parts[0] == module
                    }
                    if not any(f"{module}.{s}" in loaded for s in subs):
                        continue
                found[dist] = _md.version(dist)
            except _md.PackageNotFoundError:
                pass
    return found


def _install(packages):
    """Install `packages` without replacing what this session has already loaded.

    Colab imports numpy, pandas and friends before the first cell runs. pip
    replaces one when a dependency declares a newer version, and then the old
    copy stays in memory and the next import mixes the two (numpy's
    "_blas_supports_fpe" error). protspace itself accepts Colab's versions; its
    biocentral-api client pins tqdm>=4.68 and, through biotrainer-core,
    numpy>=2.4.1, pandas>=2.3.3 and h5py>=3.15, yet uses only biotrainer-core's
    FASTA reader. uv's --override holds every loaded distribution at the
    version in memory; plain pip is the fallback when uv cannot be had or the
    held versions cannot be satisfied, and the check below the fallback.
    """
    global _stale_after_install
    quiet = {"capture_output": True, "text": True}
    before = _loaded_distributions()
    with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as held:
        held.write("".join(f"{d}=={v}\n" for d, v in sorted(before.items())))
    run = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-qqq", "--disable-pip-version-check", "uv"], **quiet
    )
    if not run.returncode:
        run = subprocess.run(
            [sys.executable, "-m", "uv", "pip", "install", "--quiet", "--python", sys.executable,
             "--override", held.name, *packages],
            **quiet,
        )
    if run.returncode:
        run = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-qqq", "--disable-pip-version-check", *packages],
            **quiet,
        )
        if run.returncode:
            raise SystemExit(run.stdout[-3000:] + run.stderr[-3000:])
    # The safety net: anything loaded that changed on disk all the same needs a
    # restart. Kept in this session's globals: a re-run without the restart finds
    # nothing left to install, but the old copy is still the one in memory.
    after = _loaded_distributions()
    _stale_after_install = globals().get("_stale_after_install", {}) | {
        dist: f"{old} → {after[dist]}"
        for dist, old in before.items()
        if after.get(dist, old) != old
    }
    if _stale_after_install:
        raise SystemExit(
            "The install replaced packages this session had already loaded: "
            + ", ".join(f"{d} ({v})" for d, v in sorted(_stale_after_install.items()))
            + ".\nRuntime → Restart session, then run this cell again "
            "(or Run all). The second run is quick: nothing is installed again."
        )
# --- end of install helper ---

import h5py
import numpy as np

# checkpoint -> the short key ProtSpace names the embedding by (written as the
# file's model_name attribute and into its name).
MODEL_SHORT_KEYS = {
    "Rostlab/prot_t5_xl_half_uniref50-enc": "prot_t5",
    "Rostlab/ProstT5_fp16": "prost_t5",
    "ElnaggarLab/ankh-base": "ankh_base",
    "ElnaggarLab/ankh-large": "ankh_large",
    "ElnaggarLab/ankh3-large": "ankh3_large",
    "facebook/esm2_t6_8M_UR50D": "esm2_8m",
    "facebook/esm2_t12_35M_UR50D": "esm2_35m",
    "facebook/esm2_t30_150M_UR50D": "esm2_150m",
    "facebook/esm2_t33_650M_UR50D": "esm2_650m",
    "biohub/ESMC-300M": "esmc_300m",
    "biohub/ESMC-600M": "esmc_600m",
    "biohub/esm3-sm-open-v1": "esm3_open",
}

# The Preparation notebook's FASTA example (a test keeps the two equal): 20 reviewed
# entries of each of three snake-venom families, short enough to embed in a minute.
_FASTA_EXAMPLE = "example_snake_venom.fasta"
_FASTA_EXAMPLE_FAMILIES = {
    "three-finger toxin": '(family:"three-finger toxin") AND (reviewed:true)',
    "phospholipase A2": '(family:"phospholipase A2") AND (reviewed:true) AND (taxonomy_id:8570)',
    "Kunitz-type": '(family:"venom Kunitz-type") AND (reviewed:true)',
}
_FASTA_EXAMPLE_SIZE = 20

_UNIPROT_ACCESSION = re.compile(r"[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}")


def _say(label, text=""):
    print(f"{label:<10}{text}")


def _clock(seconds):
    seconds = int(seconds)
    return f"{seconds // 60}:{seconds % 60:02d}"


class _Live:
    """One output line whose elapsed time keeps counting until done()."""

    def __init__(self, label, text):
        self.label, self.text, self.t0 = label, text, time.time()
        self._stop = threading.Event()
        self._handle = None
        try:  # a live line only in a notebook kernel; plain output elsewhere
            from IPython import get_ipython
            from IPython.display import Pretty, display

            if getattr(get_ipython(), "kernel", None) is not None:
                self._pretty = Pretty
                self._handle = display(Pretty(self._line()), display_id=True)
        except Exception:
            self._handle = None
        if self._handle is not None:
            threading.Thread(target=self._tick, daemon=True).start()

    def _line(self, text=None):
        if text is not None:
            return f"{self.label:<10}{text}"
        return f"{self.label:<10}{self.text}... {_clock(time.time() - self.t0)}"

    def _tick(self):
        while not self._stop.wait(1):
            self._handle.update(self._pretty(self._line()))

    def done(self, text):
        """Replace the line with *text*, which may name the elapsed time as {time}."""
        self._stop.set()
        line = self._line(text.format(time=_clock(time.time() - self.t0)))
        if self._handle is not None:
            self._handle.update(self._pretty(line))
        else:
            print(line)


def read_fasta(path):
    """(IDs, sequences) of a FASTA file: each ID is its header's first word."""
    headers, sequences, parts = [], [], None
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if line.startswith(">"):
                if parts is not None:
                    sequences.append("".join(parts))
                headers.append(line[1:].split()[0] if line[1:].split() else "")
                parts = []
            elif line and parts is not None:
                parts.append(line)
    if parts is not None:
        sequences.append("".join(parts))
    return headers, sequences


def _protein_ids(headers):
    """(IDs, note): UniProt accessions when asked for, '/' replaced (HDF5 names cannot hold it)."""
    ids, note = list(headers), "the first word of each FASTA header"
    if should_extract_uniprot_id:
        found = [_UNIPROT_ACCESSION.search(h) for h in headers]
        ids = [m.group(0) if m else h for m, h in zip(found, headers, strict=True)]
        hits = [m.group(0) for m in found if m]
        if not hits:
            note = "the first word of each FASTA header (none holds a UniProt accession)"
        else:
            note = f"UniProt accessions, e.g. {hits[0]}"
            if len(hits) < len(ids):
                note += f" ({len(ids) - len(hits):,} headers without one keep their first word)"
    slashed = sum("/" in i for i in ids)
    if slashed:
        ids = [i.replace("/", "_") for i in ids]
        note += f"; '/' replaced by '_' in {slashed:,} of them"
    return ids, note


def _fetch_example():
    """Write the example FASTA, fetched from UniProt."""
    import urllib.request
    from urllib.parse import urlencode

    records = []
    for query in _FASTA_EXAMPLE_FAMILIES.values():
        url = "https://rest.uniprot.org/uniprotkb/search?" + urlencode(
            {"query": query, "format": "fasta", "size": _FASTA_EXAMPLE_SIZE})
        with urllib.request.urlopen(url, timeout=60) as resp:
            text = resp.read().decode()
        records.append(text if text.endswith("\n") else text + "\n")
    Path(_FASTA_EXAMPLE).write_text("".join(records))


def _choose_fasta():
    """(path, what it is): the path from the settings, an upload, or the example."""
    chosen = fasta_filename.strip()
    if chosen:
        if chosen.startswith("/content/drive") and not Path("/content/drive/MyDrive").exists():
            from google.colab import drive

            drive.mount("/content/drive")
        path = Path(chosen).expanduser()
        if not path.is_file():
            raise SystemExit(f"No file at {path}. Check the FASTA path in the settings "
                             "(Google Drive files are under /content/drive/MyDrive/...).")
        return path, ""
    if upload_fasta:
        from google.colab import files

        print("Pick a FASTA file to upload (wait for 'Saving ...').")
        uploaded = files.upload()
        if not uploaded:
            raise SystemExit("No file received. Run the cell again and pick a FASTA file, "
                             "or enter its path in the settings.")
        return Path(next(iter(uploaded))), ""
    if not Path(_FASTA_EXAMPLE).is_file():
        try:
            _fetch_example()
        except Exception as exc:
            raise SystemExit(f"UniProt could not be reached ({exc}). Run the cell again, "
                             "or enter your own FASTA's path in the settings.") from None
    families = ", ".join(_FASTA_EXAMPLE_FAMILIES)
    return Path(_FASTA_EXAMPLE), (f" (the example: {_FASTA_EXAMPLE_SIZE} Swiss-Prot entries each "
                                  f"of {families}; enter a path or tick upload for your own)")


def _load_failure(checkpoint, exc):
    """SystemExit saying why a checkpoint did not load and what to do."""
    text = f"{type(exc).__name__}: {exc}"
    if any(s in text for s in ("401", "403", "GatedRepo", "gated", "Unauthorized")):
        # huggingface_hub reads a Colab secret named HF_TOKEN by itself.
        why = ("it is gated or private: accept its licence on huggingface.co while signed "
               "in, add your token as the Colab secret HF_TOKEN (key icon, left sidebar), "
               "and run the cell again.")
    elif any(s in text for s in ("ConnectionError", "Timeout", "timed out", "Max retries")):
        why = "Hugging Face did not answer: check the connection and run the cell again."
    elif isinstance(exc, ImportError):
        why = ("a package it needs is missing: Runtime → Restart session, then run the "
               "cell again (ESM3 installs the `esm` package first).")
    else:
        why = "see the error below."
    return SystemExit(f"Could not load {checkpoint}: {why}\n{text[:1500]}")


def _weights_size(checkpoint):
    """' (2.4 GB, downloaded once per runtime)' for a model not yet downloaded, else ''."""
    try:
        from huggingface_hub import HfApi, try_to_load_from_cache

        files = {s.rfilename: s.size or 0 for s in HfApi().model_info(checkpoint, files_metadata=True).siblings}
        weights = [f for f in files if f.endswith(".safetensors")] or [f for f in files if f.endswith(".bin")]
        if not weights or all(isinstance(try_to_load_from_cache(checkpoint, f), str) for f in weights):
            return ""
        return f" ({sum(files[f] for f in weights) / 1e9:.1f} GB, downloaded once per runtime)"
    except Exception:
        return ""


def setup_model(checkpoint):
    """(model, tokenizer, model type), loaded once per checkpoint and runtime."""
    cache = globals().setdefault("_LOADED_MODEL", {})
    if checkpoint in cache:
        return cache[checkpoint]
    cache.clear()  # free the previous model's memory first
    import torch

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    from huggingface_hub.utils import disable_progress_bars
    from transformers.utils import logging as hf_logging

    disable_progress_bars()
    hf_logging.disable_progress_bar()
    # The checkpoints carry heads an encoder-only load leaves out (ESM2's and ESM C's
    # masked-LM heads, Ankh's decoder), and transformers 5 prints them as UNEXPECTED
    # on every load: the intended outcome here, so its messages are kept to errors.
    verbosity = hf_logging.get_verbosity()
    hf_logging.set_verbosity_error()
    try:
        cache[checkpoint] = _setup_model(checkpoint)
    except (SystemExit, KeyboardInterrupt):
        raise
    except Exception as exc:
        raise _load_failure(checkpoint, exc) from exc
    finally:
        hf_logging.set_verbosity(verbosity)
    return cache[checkpoint]


def _setup_model(checkpoint):
    import torch
    from transformers import AutoModel, AutoTokenizer, EsmModel, T5EncoderModel, T5Tokenizer

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if checkpoint.startswith("facebook/esm2"):
        mod_type, tokenizer = "esm_transformer", AutoTokenizer.from_pretrained(checkpoint)
        # No pooler: only the per-residue states are used, and the checkpoints
        # ship none, so it would be randomly initialised.
        model = EsmModel.from_pretrained(checkpoint, add_pooling_layer=False)
    elif checkpoint.startswith("biohub/ESMC"):
        # transformers' own ESM C (since 5.18), the official weights: no `esm`
        # package and no login.
        mod_type, tokenizer = "esmc", AutoTokenizer.from_pretrained(checkpoint)
        model = AutoModel.from_pretrained(checkpoint)
    elif checkpoint == "biohub/esm3-sm-open-v1":
        from esm.models.esm3 import ESM3  # the `esm` package, installed only for ESM3

        mod_type, tokenizer = "native_esm3", None
        model = ESM3.from_pretrained("esm3-open")
    elif checkpoint.startswith("ElnaggarLab/ankh"):
        mod_type, tokenizer = "ankh", AutoTokenizer.from_pretrained(checkpoint)
        model = T5EncoderModel.from_pretrained(checkpoint)
    elif checkpoint.startswith("Rostlab/"):
        is_prostt5 = "prostt5" in checkpoint.lower()
        mod_type = "prost_t5" if is_prostt5 else "prot_t5"
        tokenizer = T5Tokenizer.from_pretrained(checkpoint, do_lower_case=not is_prostt5)
        # Half precision on a GPU, as the checkpoints are stored; full on a CPU,
        # where half-precision matrix products are slow or missing.
        dtype = torch.float16 if device.type == "cuda" else torch.float32
        model = T5EncoderModel.from_pretrained(checkpoint, torch_dtype=dtype)
    else:
        raise ValueError(f"Unknown model checkpoint: {checkpoint}")
    return model.to(device).eval(), tokenizer, mod_type


def preprocess_sequences(sequences, mod_type):
    """Sequences as each tokenizer expects them: rare residues as X, T5 residues spaced."""
    out = [re.sub(r"[BJOUZ]", "X", s.upper()) for s in sequences]
    if mod_type in ("prot_t5", "prost_t5"):
        out = [" ".join(s) for s in out]
    if mod_type == "prost_t5":
        out = ["<AA2fold> " + s for s in out]
    return out


def _embed_native_esm3(sequence, emb_type, model):
    """One sequence through ESM3 (the `esm` package embeds one at a time)."""
    import torch
    from esm.sdk.api import ESMProtein, SamplingConfig

    tokens = model.encode(ESMProtein(sequence=sequence)).to(next(model.parameters()).device)
    with torch.inference_mode():
        out = model.forward_and_sample(tokens, SamplingConfig(return_per_residue_embeddings=True))
    per_res = out.per_residue_embedding.squeeze(0).cpu().float().numpy()[1:-1]  # strip BOS/EOS
    return per_res.mean(axis=0) if emb_type == "per_prot" else per_res


def _embed_transformers_batch(sequences, emb_type, mod_type, model, tokenizer, max_len):
    """A batch through a transformers model; special tokens stripped before pooling."""
    import torch

    inputs = tokenizer(
        sequences, return_tensors="pt", max_length=max_len + 2, truncation=True,
        padding=True, add_special_tokens=True,
    ).to(model.device)
    with torch.inference_mode():
        hidden = model(**inputs).last_hidden_state.cpu().float().numpy()
    results = []
    for i, seq_len in enumerate(inputs["attention_mask"].sum(dim=1).tolist()):
        # ESM2, ESM C and ProstT5 wrap a sequence in a start token and <eos>;
        # ProtT5 and Ankh add only <eos>.
        if mod_type in ("esm_transformer", "esmc", "prost_t5"):
            emb = hidden[i, 1 : seq_len - 1, :]
        else:
            emb = hidden[i, : seq_len - 1, :]
        results.append(emb.mean(axis=0) if emb_type == "per_prot" else emb)
    return results


def create_embedding(checkpoint, ids, sequences, emb_type="per_prot", output_file="embeddings.h5",
                     max_len=2000, batch_size=32, model_key=None):
    """Embed what *output_file* lacks; (newly written, too long, out of memory) IDs.

    ``model_key`` is stored as the file's ``model_name`` attribute, which is how
    ProtSpace names the embedding (and labels its projections) when it loads it.
    """
    import torch
    from tqdm.auto import tqdm

    model, tokenizer, mod_type = setup_model(checkpoint)
    with h5py.File(output_file, "a") as hdf:
        if model_key:
            hdf.attrs["model_name"] = model_key
        existing = set(hdf.keys())
        todo = [(i, s) for i, s in zip(ids, sequences, strict=True) if i not in existing]
        too_long = [i for i, s in todo if len(s) > max_len]
        todo = sorted(((i, s) for i, s in todo if len(s) <= max_len), key=lambda r: len(r[1]))
        oom, written = [], []
        if not todo:
            return written, too_long, oom
        bar = tqdm(total=len(todo), desc="Embedding", unit="seq")
        if mod_type == "native_esm3":
            for pid, seq in todo:
                hdf.create_dataset(pid, data=_embed_native_esm3(seq, emb_type, model).astype(np.float32))
                written.append(pid)
                bar.update(1)
        else:
            # Shortest first, so a batch size halved on out-of-memory never has to grow back.
            start, bs = 0, max(1, batch_size)
            while start < len(todo):
                batch = todo[start : start + bs]
                try:
                    vectors = _embed_transformers_batch(
                        preprocess_sequences([s for _, s in batch], mod_type),
                        emb_type, mod_type, model, tokenizer, max_len,
                    )
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    if bs > 1:
                        bs = max(1, bs // 2)
                        continue
                    oom.append(batch[0][0])
                    start += 1
                    bar.update(1)
                    continue
                for (pid, _), vec in zip(batch, vectors, strict=True):
                    hdf.create_dataset(pid, data=vec.astype(np.float32))
                    written.append(pid)
                start += len(batch)
                bar.update(len(batch))
        bar.close()
    return written, too_long, oom


def _run():
    import torch

    t0 = time.time()
    key = MODEL_SHORT_KEYS.get(model_name, model_name.replace("/", "_"))
    if model_name == "biohub/esm3-sm-open-v1":
        # The `esm` package pins transformers < 5, so it is installed only for ESM3,
        # and before transformers is first imported.
        live = _Live("Setup", "installing the `esm` package for ESM3")
        _install(["esm"])
        live.done("installed the `esm` package for ESM3 in {time}")
    if torch.cuda.is_available():
        where = f"on the GPU ({torch.cuda.get_device_name(0)})"
    else:
        where = ("on the CPU, which is slow: Runtime → Change runtime type → T4 GPU, "
                 "then run this cell again")
    kind = "per protein" if embedding_type == "per_prot" else "per residue"
    _say("Model", f"https://huggingface.co/{model_name} ('{key}'), {kind}, {where}")

    fasta_path, about = _choose_fasta()
    try:
        headers, sequences = read_fasta(fasta_path)
    except (OSError, UnicodeDecodeError) as exc:
        raise SystemExit(f"Could not read {fasta_path} ({type(exc).__name__}: {exc}). "
                         "Check that it is a FASTA file: '>' header lines, each followed "
                         "by its sequence.") from None
    if not headers:
        raise SystemExit(f"No FASTA records ('>' lines) in {fasta_path}.")
    ids, id_note = _protein_ids(headers)
    seen, dupes = set(), []
    for i in ids:
        if i in seen:
            dupes.append(i)
        seen.add(i)
    if dupes:
        raise SystemExit(f"{len(set(dupes)):,} ID(s) appear more than once, e.g. "
                         f"{', '.join(sorted(set(dupes))[:5])}: an .h5 holds one vector per "
                         "ID. Remove the duplicates from the FASTA and run the cell again.")
    _say("Proteins", f"{len(ids):,} from {fasta_path.name}{about}")
    _say("IDs", id_note)

    suffix = "" if embedding_type == "per_prot" else "_per_res"
    output_file = str(fasta_path.with_name(f"{fasta_path.stem}_{key}{suffix}.h5"))
    globals()["output_file"] = output_file

    loaded = model_name in globals().get("_LOADED_MODEL", {})
    if not loaded:
        size = _weights_size(model_name)
        live = _Live("Weights", f"downloading and loading{size}" if size else "loading")
        setup_model(model_name)
        live.done("loaded in {time}" + size)

    written, too_long, oom = create_embedding(
        model_name, ids, sequences, emb_type=embedding_type, output_file=output_file,
        max_len=max_sequence_length, batch_size=batch_size, model_key=key,
    )
    with h5py.File(output_file, "r") as h5:
        stored = len(h5)
        dim = h5[next(iter(h5))].shape[-1] if stored else 0
    size = Path(output_file).stat().st_size / 1e6
    reused = stored - len(written)
    _say("Saved", f"{stored:,} embeddings ({dim}-D) in {Path(output_file).name}, {size:.1f} MB, "
                  f"named '{key}', in {_clock(time.time() - t0)}"
                  + (f"; {reused:,} were already there" if reused else ""))
    if too_long:
        _say("", f"Skipped {len(too_long):,} longer than {max_sequence_length:,} residues "
                 f"(e.g. {', '.join(too_long[:3])}); raise max_sequence_length to include them.")
    if oom:
        _say("", f"Skipped {len(oom):,} the GPU could not fit even alone (e.g. {', '.join(oom[:3])}).")
    try:
        from google.colab import files
    except ImportError:
        _say("", f"Written to {Path(output_file).resolve()}.")
        return
    _say("", "Downloading it to your computer (it is also in the Files pane). For ProtSpace, "
             "load it in the Preparation notebook's Embeddings (.h5) tab.")
    files.download(output_file)


if __name__ == "__main__":  # Colab runs a cell as __main__; the tests import it
    _run()


In [ ]:
# @title Optional: download the .h5 again
# @markdown If the browser blocked the download, or you need the file once more.
from pathlib import Path

if not Path(globals().get("output_file", "")).is_file():
    raise SystemExit("No embeddings yet: run the cell above first.")
from google.colab import files

print(f"Downloading {Path(output_file).name} (it is also in the Files pane).")
files.download(output_file)